# Validation (c): shot-noise-limited ENOB of the AWEN photonic GEMM model vs theory

**Claim checked.** The Rust photonic GEMM model (`awen_native.photonic_gemm`,
AEP-0022) produces, under a coherently sampled sine test, the effective number
of bits that theory predicts:

* shot-noise limited: ENOB = (10·log10(N_fs/2) − 1.76)/6.02, where N_fs is the
  number of full-scale photoelectrons;
* with a b-bit ADC, Δ²/12 is added to the noise power, with Δ = 1/(2^{b−1} − 1).

ENOB follows IEEE Std 1241-2010 (doi:10.1109/IEEESTD.2011.5692956).

**Tolerance.** |ENOB_measured − ENOB_theory| < 0.1 bit, using the mean of 4
seeds of 4096 samples each.

**Fidelity.** Compact model, simulated. Not a hardware measurement.

In [ ]:
import numpy as np
from awen_physics.enob import shot_only_contract, sine_test, theory_enob

print("photons/MAC  ADC bits  N_fs        ENOB sim  ENOB theory")
rows = []
for photons in (10, 100, 1000, 10000, 100000):
    for adc in (None, 8):
        runs = [sine_test(shot_only_contract(photons, adc), seed=s) for s in range(4)]
        measured = float(np.mean([r["enob"] for r in runs]))
        n_fs = runs[0]["full_scale_photoelectrons"]
        theory = theory_enob(n_fs, adc)
        rows.append((photons, adc, n_fs, measured, theory))
        print(f"{photons:>11}  {str(adc):>8}  {n_fs:<10.0f}  {measured:8.3f}  {theory:11.3f}")
        assert abs(measured - theory) < 0.1, (photons, adc, measured, theory)

In [ ]:
# Each 10x in photons per MAC buys 10·log10(10)/6.02 = 1.66 bits while shot noise dominates.
shot = [r for r in rows if r[1] is None]
steps = [b[3] - a[3] for a, b in zip(shot, shot[1:])]
print("ENOB gained per decade of photons:", [round(s, 3) for s in steps])
assert all(abs(s - 10 / 6.02) < 0.1 for s in steps)

**Result.** The simulator matches shot-noise theory within 0.1 bit over four
decades of optical power. With an 8-bit ADC the curve saturates near 8 bits,
as the quantization term predicts. This validates the model's internal
consistency only. Real photodetectors add noise sources that are not modeled
here, which AEP-0022 lists under its limits.